# 03 — Track Re-identification สำหรับ video001

Notebook นี้ใช้ผล Face Detection ที่มีอยู่แล้ว เพื่อรวม `track_id` ที่แตกเป็นจำนวนมากให้กลับมาเป็น `person_id` ของคนจริง

Input:
- `dataset/yolo_face_detection/video001/detection.csv`
- `dataset/raw_videos/video001.webm`

Output:
- `dataset/reid/video001/track_samples/`
- `dataset/reid/video001/track_embeddings.npy`
- `dataset/reid/video001/track_embedding_index.csv`
- `dataset/reid/video001/track_person_map.csv`
- `dataset/reid/video001/reidentified_detection.csv`
- `dataset/reid/video001/person_summary.csv`

แนวคิด:
1. เลือกตัวอย่างหลายเฟรมต่อ track
2. Crop ใบหน้าจากวิดีโอ
3. ใช้ InsightFace recognition model สร้าง embedding
4. เฉลี่ย embedding ต่อ track
5. Cluster track embeddings
6. เพิ่ม `person_id` กลับเข้า detection.csv

> ไม่ต้อง Face Detect ใหม่ทั้งวิดีโอ

## Cell 1 — Path / Cache

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

VIDEO_PATH = PROJECT_ROOT / "dataset" / "raw_videos" / f"{VIDEO_ID}.webm"
DETECTION_CSV = PROJECT_ROOT / "dataset" / "yolo_face_detection" / VIDEO_ID / "detection.csv"

OUTPUT_ROOT = PROJECT_ROOT / "dataset" / "reid" / VIDEO_ID
SAMPLE_ROOT = OUTPUT_ROOT / "track_samples"
CACHE_ROOT = PROJECT_ROOT / "cache"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")

print("VIDEO_PATH    :", VIDEO_PATH)
print("DETECTION_CSV :", DETECTION_CSV)
print("OUTPUT_ROOT   :", OUTPUT_ROOT)

if not VIDEO_PATH.exists():
    raise FileNotFoundError(VIDEO_PATH)

if not DETECTION_CSV.exists():
    raise FileNotFoundError(DETECTION_CSV)

## Cell 2 — Environment

In [ ]:
import sys
import numpy as np
import pandas as pd

print("Python :", sys.executable)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)

## Cell 3 — Install dependencies

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "insightface==1.0.1",
    "onnxruntime-gpu==1.29.0",
    "scikit-learn",
    "opencv-python",
    "pandas",
    "numpy",
    "matplotlib",
    "tqdm"
])

print("Dependencies ready")

## Cell 4 — Load detection.csv

In [ ]:
det_df = pd.read_csv(DETECTION_CSV)

required_cols = {
    "frame_index", "track_id",
    "x1", "y1", "x2", "y2",
    "det_score"
}

missing = required_cols - set(det_df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

det_df["frame_index"] = det_df["frame_index"].astype(int)
det_df["track_id"] = det_df["track_id"].astype(int)

print("Detection rows :", len(det_df))
print("Tracks         :", det_df["track_id"].nunique())
print("Max frame      :", det_df["frame_index"].max())

track_stats = (
    det_df.groupby("track_id")
    .agg(
        detections=("frame_index", "count"),
        first_frame=("frame_index", "min"),
        last_frame=("frame_index", "max"),
        mean_x1=("x1", "mean"),
        mean_y1=("y1", "mean"),
        mean_x2=("x2", "mean"),
        mean_y2=("y2", "mean"),
        mean_conf=("det_score", "mean"),
    )
    .reset_index()
)

track_stats["span"] = (
    track_stats["last_frame"] - track_stats["first_frame"] + 1
)
track_stats["coverage"] = (
    track_stats["detections"] / track_stats["span"]
)

track_stats = track_stats.sort_values(
    "detections", ascending=False
).reset_index(drop=True)

display(track_stats.head(30))

## Cell 5 — Select sample frames per track

In [ ]:
SAMPLES_PER_TRACK = 7
MIN_TRACK_DETECTIONS = 5

sample_rows = []

for track_id, group in det_df.groupby("track_id"):
    group = group.sort_values("frame_index").reset_index(drop=True)

    if len(group) < MIN_TRACK_DETECTIONS:
        continue

    n = min(SAMPLES_PER_TRACK, len(group))

    sample_positions = np.linspace(
        0,
        len(group) - 1,
        n
    ).astype(int)

    selected = group.iloc[np.unique(sample_positions)]

    for sample_rank, (_, row) in enumerate(selected.iterrows()):
        sample_rows.append({
            "track_id": int(track_id),
            "sample_rank": int(sample_rank),
            "frame_index": int(row["frame_index"]),
            "x1": float(row["x1"]),
            "y1": float(row["y1"]),
            "x2": float(row["x2"]),
            "y2": float(row["y2"]),
            "det_score": float(row["det_score"]),
        })

samples_df = pd.DataFrame(sample_rows)

print("Sample rows:", len(samples_df))
print("Tracks sampled:", samples_df["track_id"].nunique())

display(samples_df.head(20))

## Cell 6 — Crop face samples

In [ ]:
import cv2
from tqdm.auto import tqdm

FACE_MARGIN = 0.20
MIN_FACE_SIZE = 60

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"เปิดวิดีโอไม่ได้: {VIDEO_PATH}")

sample_meta = []

for _, row in tqdm(
    samples_df.iterrows(),
    total=len(samples_df),
    desc="cropping track samples"
):
    frame_index = int(row["frame_index"])
    track_id = int(row["track_id"])

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ret, frame = cap.read()

    if not ret or frame is None:
        continue

    h, w = frame.shape[:2]

    x1, y1, x2, y2 = map(
        float,
        [row["x1"], row["y1"], row["x2"], row["y2"]]
    )

    bw = x2 - x1
    bh = y2 - y1

    margin_x = bw * FACE_MARGIN
    margin_y = bh * FACE_MARGIN

    cx1 = max(0, int(round(x1 - margin_x)))
    cy1 = max(0, int(round(y1 - margin_y)))
    cx2 = min(w, int(round(x2 + margin_x)))
    cy2 = min(h, int(round(y2 + margin_y)))

    crop = frame[cy1:cy2, cx1:cx2]

    if crop.size == 0:
        continue

    ch, cw = crop.shape[:2]
    if min(ch, cw) < MIN_FACE_SIZE:
        continue

    track_dir = SAMPLE_ROOT / f"track_{track_id:03d}"
    track_dir.mkdir(parents=True, exist_ok=True)

    sample_path = track_dir / f"frame_{frame_index:08d}.jpg"

    cv2.imwrite(
        str(sample_path),
        crop,
        [cv2.IMWRITE_JPEG_QUALITY, 90]
    )

    sample_meta.append({
        "track_id": track_id,
        "frame_index": frame_index,
        "sample_path": str(sample_path.relative_to(PROJECT_ROOT)),
        "det_score": float(row["det_score"]),
    })

cap.release()

sample_meta_df = pd.DataFrame(sample_meta)

SAMPLE_META_CSV = OUTPUT_ROOT / "track_sample_metadata.csv"
sample_meta_df.to_csv(
    SAMPLE_META_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Saved samples:", len(sample_meta_df))
print("CSV:", SAMPLE_META_CSV)

display(sample_meta_df.head(20))

## Cell 7 — Load InsightFace

In [ ]:
import onnxruntime as ort
from insightface.app import FaceAnalysis

available = ort.get_available_providers()
print("Available providers:", available)

providers = (
    ["CUDAExecutionProvider", "CPUExecutionProvider"]
    if "CUDAExecutionProvider" in available
    else ["CPUExecutionProvider"]
)

app = FaceAnalysis(
    name="buffalo_l",
    providers=providers
)

app.prepare(
    ctx_id=0 if "CUDAExecutionProvider" in providers else -1,
    det_size=(640, 640)
)

print("InsightFace ready")

## Cell 8 — Sample embeddings

In [ ]:
embedding_rows = []
embedding_vectors = []

for _, row in tqdm(
    sample_meta_df.iterrows(),
    total=len(sample_meta_df),
    desc="face embeddings"
):
    image_path = PROJECT_ROOT / row["sample_path"]
    image = cv2.imread(str(image_path))

    if image is None:
        continue

    faces = app.get(image)

    if not faces:
        continue

    face = max(
        faces,
        key=lambda f: float(f.det_score)
    )

    emb = np.asarray(
        face.normed_embedding,
        dtype=np.float32
    ).reshape(-1)

    embedding_rows.append({
        "track_id": int(row["track_id"]),
        "frame_index": int(row["frame_index"]),
        "sample_path": row["sample_path"],
        "face_det_score": float(face.det_score),
        "embedding_index": len(embedding_vectors),
    })

    embedding_vectors.append(emb)

sample_embedding_df = pd.DataFrame(embedding_rows)

sample_embeddings = (
    np.stack(embedding_vectors)
    if embedding_vectors
    else np.empty((0, 512), dtype=np.float32)
)

print("Embedding samples:", len(sample_embedding_df))
print("Embedding shape  :", sample_embeddings.shape)

display(sample_embedding_df.head(20))

## Cell 9 — Track-level embeddings

In [ ]:
track_embedding_rows = []
track_embedding_vectors = []

for track_id, group in sample_embedding_df.groupby("track_id"):
    indices = group["embedding_index"].astype(int).to_numpy()
    embs = sample_embeddings[indices]

    mean_emb = embs.mean(axis=0)
    norm = np.linalg.norm(mean_emb)

    if norm > 0:
        mean_emb = mean_emb / norm

    track_embedding_rows.append({
        "track_id": int(track_id),
        "num_embedding_samples": len(embs),
        "track_embedding_index": len(track_embedding_vectors),
    })

    track_embedding_vectors.append(
        mean_emb.astype(np.float32)
    )

track_embedding_df = pd.DataFrame(track_embedding_rows)

track_embeddings = (
    np.stack(track_embedding_vectors)
    if track_embedding_vectors
    else np.empty((0, 512), dtype=np.float32)
)

print("Tracks with embeddings:", len(track_embedding_df))
print("Track embedding shape :", track_embeddings.shape)

display(track_embedding_df.head(20))

## Cell 10 — Save embeddings

In [ ]:
TRACK_EMB_NPY = OUTPUT_ROOT / "track_embeddings.npy"
TRACK_EMB_INDEX_CSV = OUTPUT_ROOT / "track_embedding_index.csv"
SAMPLE_EMB_INDEX_CSV = OUTPUT_ROOT / "sample_embedding_index.csv"

np.save(TRACK_EMB_NPY, track_embeddings)

track_embedding_df.to_csv(
    TRACK_EMB_INDEX_CSV,
    index=False,
    encoding="utf-8-sig"
)

sample_embedding_df.to_csv(
    SAMPLE_EMB_INDEX_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(" -", TRACK_EMB_NPY)
print(" -", TRACK_EMB_INDEX_CSV)
print(" -", SAMPLE_EMB_INDEX_CSV)

## Cell 11 — Similarity distribution

In [ ]:
similarity_matrix = track_embeddings @ track_embeddings.T

upper_values = similarity_matrix[
    np.triu_indices_from(similarity_matrix, k=1)
]

print(
    pd.Series(upper_values).describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

## Cell 12 — Cluster tracks

In [ ]:
from sklearn.cluster import AgglomerativeClustering

DISTANCE_THRESHOLD = 0.35

clustering = AgglomerativeClustering(
    n_clusters=None,
    metric="cosine",
    linkage="average",
    distance_threshold=DISTANCE_THRESHOLD
)

person_labels = clustering.fit_predict(track_embeddings)

track_person_df = track_embedding_df.copy()
track_person_df["person_id"] = person_labels.astype(int)

person_order = (
    track_person_df.groupby("person_id")["track_id"]
    .min()
    .sort_values()
    .index
    .tolist()
)

person_remap = {
    old_id: new_id
    for new_id, old_id in enumerate(person_order)
}

track_person_df["person_id"] = (
    track_person_df["person_id"]
    .map(person_remap)
    .astype(int)
)

print("Estimated persons:", track_person_df["person_id"].nunique())

display(
    track_person_df.sort_values(
        ["person_id", "track_id"]
    ).head(100)
)

## Cell 13 — Merge track stats

In [ ]:
track_person_df = track_person_df.merge(
    track_stats,
    on="track_id",
    how="left"
)

display(
    track_person_df[
        [
            "person_id",
            "track_id",
            "detections",
            "first_frame",
            "last_frame",
            "mean_x1",
            "mean_conf",
            "num_embedding_samples",
        ]
    ].sort_values(
        ["person_id", "first_frame"]
    ).head(100)
)

## Cell 14 — Person summary

In [ ]:
person_summary = (
    track_person_df
    .groupby("person_id")
    .agg(
        num_tracks=("track_id", "count"),
        total_detections=("detections", "sum"),
        first_frame=("first_frame", "min"),
        last_frame=("last_frame", "max"),
        mean_conf=("mean_conf", "mean"),
    )
    .reset_index()
    .sort_values(
        "total_detections",
        ascending=False
    )
)

display(person_summary)

## Cell 15 — Preview samples per person

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

MAX_PERSONS_TO_SHOW = min(
    10,
    track_person_df["person_id"].nunique()
)

for person_id in sorted(
    track_person_df["person_id"].unique()
)[:MAX_PERSONS_TO_SHOW]:

    person_tracks = track_person_df[
        track_person_df["person_id"] == person_id
    ]["track_id"].tolist()

    person_samples = sample_meta_df[
        sample_meta_df["track_id"].isin(person_tracks)
    ].head(12)

    print(
        f"Person {person_id} | "
        f"tracks={person_tracks[:20]}"
    )

    if len(person_samples) == 0:
        continue

    cols = 4
    rows = int(np.ceil(len(person_samples) / cols))

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(12, 3 * rows)
    )

    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, (_, sample) in zip(
        axes,
        person_samples.iterrows()
    ):
        img_path = PROJECT_ROOT / sample["sample_path"]
        img = Image.open(img_path)

        ax.imshow(img)
        ax.set_title(
            f"T{int(sample['track_id'])} "
            f"F{int(sample['frame_index'])}"
        )
        ax.axis("off")

    plt.suptitle(f"Person {person_id}")
    plt.tight_layout()
    plt.show()

## Cell 16 — Save person map

In [ ]:
TRACK_PERSON_MAP_CSV = OUTPUT_ROOT / "track_person_map.csv"
PERSON_SUMMARY_CSV = OUTPUT_ROOT / "person_summary.csv"

track_person_df.to_csv(
    TRACK_PERSON_MAP_CSV,
    index=False,
    encoding="utf-8-sig"
)

person_summary.to_csv(
    PERSON_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(" -", TRACK_PERSON_MAP_CSV)
print(" -", PERSON_SUMMARY_CSV)

## Cell 17 — Add person_id back to detection.csv

In [ ]:
reid_df = det_df.merge(
    track_person_df[
        ["track_id", "person_id"]
    ],
    on="track_id",
    how="left"
)

REID_DETECTION_CSV = OUTPUT_ROOT / "reidentified_detection.csv"

reid_df.to_csv(
    REID_DETECTION_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Rows:", len(reid_df))
print(
    "Person IDs:",
    sorted(
        reid_df["person_id"]
        .dropna()
        .unique()
        .tolist()
    )
)
print("Saved:", REID_DETECTION_CSV)

display(reid_df.head(20))

In [ ]:
# ============================================================
# QA — ตรวจ Person ID
# ============================================================

print("จำนวน Person:", reid_df["person_id"].nunique())

person_stats = (
    reid_df
    .dropna(subset=["person_id"])
    .groupby("person_id")
    .agg(
        detections=("frame_index", "count"),
        tracks=("track_id", "nunique"),
        first_frame=("frame_index", "min"),
        last_frame=("frame_index", "max"),
        mean_x1=("x1", "mean"),
        mean_x2=("x2", "mean"),
        mean_conf=("det_score", "mean"),
    )
    .reset_index()
)

person_stats["person_id"] = (
    person_stats["person_id"].astype(int)
)

display(person_stats)

# ขั้นต่อไป

เมื่อ QA แล้วว่า `person_id` ถูกต้อง ให้ใช้:

`dataset/reid/video001/reidentified_detection.csv`

แทน `detection.csv` เดิมใน Mouth Extraction

จากนั้น Mouth Extraction จะจัดข้อมูลตาม `person_id` แทน `track_id`